# Notebook 03c — AH-PC Per-Subject Pre-Training (k=1,2,4,8)

Notebooks 03 and 03b pre-train **one global model** on a pooled subject-level train/val
split (subjects 1-13 train, 14-16 val). This notebook instead pre-trains **a separate
model per subject per fold** — exactly like Sheibani's own protocol, and matching the
per-subject-per-fold structure `04_finetune.ipynb`/`04c_finetune_k1_raw.ipynb` already
use for fine-tuning.

## Why per-subject-per-fold pre-training?

EEG signals are highly subject-specific — the same emotion can produce quite different
raw feature patterns across people. A single global encoder has to compromise across all
16 subjects at once. Training a fresh encoder from scratch per (subject, fold) lets each
model specialise entirely to one subject's EEG patterns, using only that subject's own
data. This costs 48 separate training runs instead of 1, but each one is much smaller
and faster (only two of a subject's three sessions' worth of pairs per fold).

## Protocol

For each of the 16 subjects, for each of the 3 leave-one-session-out folds (identical
fold convention to `04_finetune.ipynb`'s trial-based split — train on two sessions, hold
out the third as val):

- Train a **fresh** `AHPCModel` (multi-horizon, raw feature space, Horizon Attention
  Module + 4 ConvTranspose2d decoder heads — identical architecture to
  `03_pretrain.ipynb`, grafted onto 03c's per-subject-per-fold training loop) from scratch
- 100 epochs, early stopping patience=15, AdamW lr=1e-3, cosine annealing, batch=64
- Save the best-val-loss checkpoint to
  `src/models/per_subject_k4/ahpc_subject{sid:02d}_fold{fold}_best.pt`

This produces 48 checkpoints (16 subjects × 3 folds), each specialised to one subject's
EEG patterns for one fold's train/val split — a natural pre-training counterpart to
`04_finetune.ipynb`'s per-subject-per-fold classifiers.

`HORIZONS = [1, 2, 4, 8]` (`MAX_HORIZON = 8`) — full multi-horizon routing via the
Horizon Attention Module, one `HorizonDecoder` per horizon. This supersedes the
`HORIZONS = [1]` single-horizon scope this notebook started with (see
`per_subject/` for those k=1-only checkpoints) — the per-subject pair files already
carry targets for all four horizons, so no data-pipeline changes were needed.

## Smoke test

`RUN_FULL_TRAINING = False` by default — the full protocol (48 runs × up to 100 epochs)
is expensive and left un-launched. A 2-epoch smoke test on **Subject 1, Fold 1** runs
automatically to confirm the per-subject data loading and training loop work, printing
train/val loss and mean horizon-attention weights per epoch.

In [1]:
import os
import math
import pickle
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:   {device}")

DATA_DIR    = "/workspaces/AH-PC/src/data/processed"
CKPT_DIR    = "/workspaces/AH-PC/src/models/per_subject_k4"
NUM_SUBJECTS = 16
os.makedirs(CKPT_DIR, exist_ok=True)

PyTorch 2.12.1+cu130
CUDA available: False
Using device:   cpu


---
## 1. Model Definition

Identical to `03_pretrain.ipynb`: full multi-horizon routing via the Horizon Attention
Module, with one ConvTranspose2d decoder head per horizon (k=1, 2, 4, 8).

- **Sheibani Encoder** (untouched): BandAttention + ChannelAttention + CNN + Bottleneck → (B, 64, 15, 5)
- **Horizon Attention Module**: Flatten(4800) → Linear(4800→64) + LN + GELU + Linear(64→4) + Softmax → α (B, 4)
- **4 × Decoder Heads** (ConvTranspose2d, one per horizon): bottleneck (B,64,15,5) → ConvTranspose2d(64→32,k=(2,1),s=(2,1)) → ReLU → ConvTranspose2d(32→16,k=(2,1),s=(2,1)) → ReLU → Conv2d(16→1,k=3) → Upsample(62,5) → prediction (B, 62, 5)
- **Projector** (fine-tuning use only, via `get_embedding()` — **not** part of the pre-training loss): Flatten(4800) → Linear(4800→256) + LayerNorm → z_t (B, 256)

In [2]:
# ── Sheibani's attention modules (reproduced exactly, identical to Notebooks 03/03b) ─

class BandAttention(nn.Module):
    def __init__(self, num_bands):
        super().__init__()
        self.fc1 = nn.Linear(num_bands * 2, num_bands // 2)
        self.fc2 = nn.Linear(num_bands // 2, num_bands)

    def forward(self, x):  # x: (B, C, num_bands)
        avg_pool = x.mean(dim=1)
        max_pool = x.max(dim=1).values
        context  = torch.cat([avg_pool, max_pool], dim=-1)
        weights  = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(1)


class ChannelAttention(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.fc1 = nn.Linear(num_channels * 2, num_channels // 2)
        self.fc2 = nn.Linear(num_channels // 2, num_channels)

    def forward(self, x):  # x: (B, C, num_bands)
        avg_pool = x.mean(dim=2)
        max_pool = x.max(dim=2).values
        context  = torch.cat([avg_pool, max_pool], dim=-1)
        weights  = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(2)


class SheibaniEncoder(nn.Module):
    """Attention-gated CNN encoder. Returns (B, 64, 15, 5) feature map."""
    OUT_CHANNELS = 64

    def __init__(self):
        super().__init__()
        self.band_attn    = BandAttention(num_bands=5)
        self.channel_attn = ChannelAttention(num_channels=62)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

    def forward(self, x):  # x: (B, 62, 5)
        x = self.band_attn(x)
        x = self.channel_attn(x)
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.bottleneck(x)
        return x  # (B, 64, 15, 5)


HORIZONS        = [1, 2, 4, 8]  # full multi-horizon routing (was [1] only)
MAX_HORIZON     = 8             # max(HORIZONS) — per-subject pair files already carry targets up to k=8
LATENT_DIM      = 256           # only used by get_embedding's projector (fine-tuning)
BOTTLENECK_FLAT = 64 * 15 * 5   # 4800 — flattened (C, H, W) bottleneck


class HorizonAttentionModule(nn.Module):
    """Soft routing: outputs len(HORIZONS) horizon weights that sum to 1, from the flattened bottleneck."""
    def __init__(self, in_dim=BOTTLENECK_FLAT, num_horizons=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, num_horizons),
        )
        # Start the softmax near-uniform: small weights + zero bias mean the
        # pre-softmax logits are all ~0 at init, so no horizon has a head start.
        nn.init.zeros_(self.net[-1].bias)
        nn.init.uniform_(self.net[-1].weight, -0.01, 0.01)

    def forward(self, bottleneck):  # bottleneck: (B, 64, 15, 5)
        return F.softmax(self.net(bottleneck), dim=-1)  # (B, num_horizons)


class HorizonDecoder(nn.Module):
    """Single-horizon decoder: bottleneck -> predicted raw segment at t+k, (B, 62, 5)."""
    def __init__(self):
        super().__init__()
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=(2, 1), stride=(2, 1)),  # (B,32,30,5)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=(2, 1), stride=(2, 1)),  # (B,16,60,5)
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),                    # (B,1,60,5)
        )
        self.upsample = nn.Upsample(size=(62, 5), mode="bilinear", align_corners=False)

    def forward(self, bottleneck):  # bottleneck: (B, 64, 15, 5)
        x = self.deconv(bottleneck)
        x = self.upsample(x)        # (B, 1, 62, 5)
        return x.squeeze(1)         # (B, 62, 5)


class AHPCModel(nn.Module):
    """Same encoder/decoder/projector family as 03/03b — now with the Horizon Attention
    Module and one ConvTranspose2d decoder head per horizon (K=4), matching 03_pretrain.ipynb."""
    def __init__(self, latent_dim=LATENT_DIM, horizons=HORIZONS):
        super().__init__()
        self.horizons = horizons
        self.sheibani_encoder = SheibaniEncoder()
        self.horizon_attn = HorizonAttentionModule(BOTTLENECK_FLAT, num_horizons=len(horizons))
        self.decoders     = nn.ModuleList([HorizonDecoder() for _ in horizons])
        # Projector is used only by get_embedding() for fine-tuning — it plays no part
        # in the pre-training loss, which is computed directly in raw feature space.
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(BOTTLENECK_FLAT, latent_dim),
            nn.LayerNorm(latent_dim),
        )

    def get_embedding(self, x):  # x: (B, 62, 5) -> (B, 256). Fine-tuning use only.
        bottleneck = self.sheibani_encoder(x)
        return self.projector(bottleneck)

    def forward(self, anchor):   # anchor: (B, 62, 5)
        bottleneck = self.sheibani_encoder(anchor)                # (B, 64, 15, 5)
        alpha      = self.horizon_attn(bottleneck)                 # (B, len(horizons))
        preds      = {k: self.decoders[i](bottleneck) for i, k in enumerate(self.horizons)}
        return bottleneck, alpha, preds


LAMBDA_ENTROPY = 0.0  # entropy is logged, not optimised — matches 03_pretrain.ipynb


def compute_loss(model, anchor, targets, device, lambda_entropy=LAMBDA_ENTROPY):
    """
    Multi-horizon weighted MSE loss in raw feature space (no entropy regularisation),
    identical to 03_pretrain.ipynb's compute_loss.

    Args:
        model:   AHPCModel in train or eval mode
        anchor:  (B, 62, 5) on device
        targets: dict {k: (B, 62, 5) on device} for k in HORIZONS — the actual raw
                 future segments (no encoder, no stop-gradient needed)

    Returns:
        loss:      scalar tensor  (call .backward() during training)
        pred_loss: float          (prediction component, for logging — equals loss here)
        entropy:   float          (H(alpha), for logging only — not in loss)
        alpha:     (B, len(HORIZONS)) detached tensor (for alpha tracking)
    """
    bottleneck, alpha, preds = model(anchor)

    # Per-sample weighted MSE (raw feature space) summed over horizons, averaged over batch
    loss_pred = torch.zeros(anchor.size(0), device=device)
    for i, k in enumerate(model.horizons):
        mse_k = F.mse_loss(preds[k], targets[k], reduction="none").mean(dim=(1, 2))
        loss_pred = loss_pred + alpha[:, i] * mse_k
    loss_pred = loss_pred.mean()  # scalar

    # Entropy H(alpha) computed for logging/tracking only — not added to the loss
    entropy = -(alpha * (alpha + 1e-8).log()).sum(dim=-1).mean()

    loss = loss_pred
    return loss, loss_pred.item(), entropy.item(), alpha.detach()


# Quick sanity check
_m = AHPCModel()
_x = torch.randn(4, 62, 5)
with torch.no_grad():
    _bn, _a, _p = _m(_x)
    _emb = _m.get_embedding(_x)
print(f"Model OK  |  bottleneck: {_bn.shape}  alpha: {_a.shape}  pred[k=1]: {_p[1].shape}")
print(f"get_embedding (fine-tuning only): {_emb.shape}")
print(f"Total params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
print(f"Initial alpha (should be near-uniform): {_a[0].numpy().round(4)}")
del _m, _x, _bn, _a, _p, _emb

Model OK  |  bottleneck: torch.Size([4, 64, 15, 5])  alpha: torch.Size([4, 4])  pred[k=1]: torch.Size([4, 62, 5])
get_embedding (fine-tuning only): torch.Size([4, 256])
Total params: 1,587,888
Initial alpha (should be near-uniform): [0.2552 0.2458 0.2527 0.2463]


---
## 2. Per-Subject Trial-Based Fold Splitting

Identical fold convention to `04_finetune.ipynb`/`04c_finetune_k1_raw.ipynb`: each
subject's 45 trials are 3 sessions of 15 trials each, laid out contiguously as
`[session 1: 0-14][session 2: 15-29][session 3: 30-44]`. Each fold holds out one
session's trials as **val** and trains on the other two:

- **Fold 1:** train = sessions 2+3 (trials 15-44), val = session 1 (trials 0-14)
- **Fold 2:** train = sessions 1+3 (trials 0-14, 30-44), val = session 2 (trials 15-29)
- **Fold 3:** train = sessions 1+2 (trials 0-29), val = session 3 (trials 30-44)

Unlike Notebooks 03/03b, there is no cross-subject split here at all — train and val
pairs both come from the **same subject**, split only by which session they belong to.

In [3]:
SESSION_TRIALS = [list(range(0, 15)), list(range(15, 30)), list(range(30, 45))]


def get_fold_trials(fold_idx: int):
    """fold_idx in {0, 1, 2}. Returns (train_trials, val_trials) for that fold,
    holding out one session's trials as val and training on the other two."""
    val_trials = SESSION_TRIALS[fold_idx]
    train_trials = [t for i, s in enumerate(SESSION_TRIALS) if i != fold_idx for t in s]
    return sorted(train_trials), sorted(val_trials)


def load_subject_pairs(subject_id: int, data_dir=DATA_DIR):
    path = os.path.join(data_dir, f"subject_{subject_id:02d}_pairs.pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def get_subject_fold_pairs(subject_id: int, fold_idx: int):
    """Split one subject's pairs into train/val by trial index for the given fold."""
    pairs = load_subject_pairs(subject_id)
    train_trials, val_trials = get_fold_trials(fold_idx)
    train_set, val_set = set(train_trials), set(val_trials)

    train_pairs = [p for p in pairs if p["trial"] in train_set]
    val_pairs   = [p for p in pairs if p["trial"] in val_set]
    return train_pairs, val_pairs, train_trials, val_trials


print("Trial-based fold splitting defined.")
for fold_idx in range(3):
    train_trials, val_trials = get_fold_trials(fold_idx)
    print(f"  Fold {fold_idx + 1}: train trials={train_trials[0]}-{train_trials[-1]} "
          f"(n={len(train_trials)}), val trials={val_trials[0]}-{val_trials[-1]} "
          f"(n={len(val_trials)})")

Trial-based fold splitting defined.
  Fold 1: train trials=15-44 (n=30), val trials=0-14 (n=15)
  Fold 2: train trials=0-44 (n=30), val trials=15-29 (n=15)
  Fold 3: train trials=0-29 (n=30), val trials=30-44 (n=15)


In [4]:
class MultiHorizonPairDataset(Dataset):
    """
    Wraps an already-filtered list of pair dicts (one subject's train or val pairs
    for one fold). Returns: (anchor, t1, t2, t4, t8, label) — all tensors. Labels are
    carried along but not used during pre-training.
    """

    def __init__(self, pairs):
        self.samples = pairs

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p      = self.samples[idx]
        anchor = torch.tensor(p["anchor"],     dtype=torch.float32)  # (62, 5)
        t1     = torch.tensor(p["targets"][1], dtype=torch.float32)  # (62, 5)
        t2     = torch.tensor(p["targets"][2], dtype=torch.float32)  # (62, 5)
        t4     = torch.tensor(p["targets"][4], dtype=torch.float32)  # (62, 5)
        t8     = torch.tensor(p["targets"][8], dtype=torch.float32)  # (62, 5)
        label  = torch.tensor(p["label"],      dtype=torch.long)
        return anchor, t1, t2, t4, t8, label


# Verify on Subject 1, Fold 1
_train_pairs, _val_pairs, _, _ = get_subject_fold_pairs(1, 0)
_train_ds = MultiHorizonPairDataset(_train_pairs)
_val_ds   = MultiHorizonPairDataset(_val_pairs)
print(f"Subject 1, Fold 1  |  train pairs: {len(_train_ds):,}  val pairs: {len(_val_ds):,}")
_a, _t1, _t2, _t4, _t8, _lbl = _train_ds[0]
print(f"Sample shapes  |  anchor: {tuple(_a.shape)}  t1: {tuple(_t1.shape)}  "
      f"t8: {tuple(_t8.shape)}  label: {_lbl.item()}")
del _train_pairs, _val_pairs, _train_ds, _val_ds, _a, _t1, _t2, _t4, _t8, _lbl

Subject 1, Fold 1  |  train pairs: 902  val pairs: 561
Sample shapes  |  anchor: (62, 5)  t1: (62, 5)  t8: (62, 5)  label: 2


---
## 3. Per-Subject-Per-Fold Training Function

`train_one_subject_fold(subject_id, fold_idx, ...)` builds that (subject, fold)'s
train/val split, trains a **fresh** `AHPCModel` from scratch (no weight sharing across
subjects or folds), and saves the best-val-loss checkpoint. Hyperparameters match
Notebooks 03/03b exactly:

- **Optimizer:** AdamW, `lr=1e-3`, `weight_decay=1e-4`
- **LR schedule:** linear warmup (5% of steps) + cosine annealing, stepped per batch
- **Batch size:** 64 · **Gradient clipping:** max norm 1.0
- **Epochs:** up to 100, early stopping patience=15 (disabled when `smoke_test=True`)
- **Loss:** multi-horizon weighted MSE (Section 1's `compute_loss`) — the per-epoch mean
  Horizon Attention weights `[k=1, k=2, k=4, k=8]` are logged alongside train/val loss

Checkpoint path: `src/models/per_subject_k4/ahpc_subject{sid:02d}_fold{fold}_best.pt`

In [5]:
def train_one_subject_fold(subject_id, fold_idx, epochs=100, batch_size=64, lr=1e-3,
                            weight_decay=1e-4, patience=15, max_grad_norm=1.0,
                            device=device, smoke_test=False, verbose=True):
    train_pairs, val_pairs, train_trials, val_trials = get_subject_fold_pairs(subject_id, fold_idx)

    train_ds = MultiHorizonPairDataset(train_pairs)
    val_ds   = MultiHorizonPairDataset(val_pairs)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = AHPCModel(latent_dim=LATENT_DIM, horizons=HORIZONS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps  = max(1, epochs * len(train_loader))
    warmup_steps = max(1, int(0.05 * total_steps))

    def _lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

    train_losses, val_losses, alpha_history = [], [], []
    best_val_loss    = float("inf")
    patience_counter = 0
    ckpt_path = os.path.join(CKPT_DIR, f"ahpc_subject{subject_id:02d}_fold{fold_idx + 1}_best.pt")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_train_loss = 0.0
        for anchor, t1, t2, t4, t8, _ in train_loader:
            anchor  = anchor.to(device)
            targets = {1: t1.to(device), 2: t2.to(device),
                       4: t4.to(device), 8: t8.to(device)}
            optimizer.zero_grad()
            loss, pred_loss, _, _ = compute_loss(model, anchor, targets, device)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            epoch_train_loss += pred_loss
        avg_train = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train)

        model.eval()
        epoch_val_loss = 0.0
        epoch_alphas = []
        with torch.no_grad():
            for anchor, t1, t2, t4, t8, _ in val_loader:
                anchor  = anchor.to(device)
                targets = {1: t1.to(device), 2: t2.to(device),
                           4: t4.to(device), 8: t8.to(device)}
                _, pred_loss, _, alpha = compute_loss(model, anchor, targets, device)
                epoch_val_loss += pred_loss
                epoch_alphas.append(alpha.cpu())
        avg_val    = epoch_val_loss / len(val_loader)
        mean_alpha = torch.cat(epoch_alphas, dim=0).mean(dim=0).numpy()
        val_losses.append(avg_val)
        alpha_history.append(mean_alpha)

        if verbose:
            lr_now = scheduler.get_last_lr()[0]
            a = mean_alpha
            print(f"    Epoch {epoch:3d}/{epochs}  train={avg_train:.5f}  "
                  f"val={avg_val:.5f}  lr={lr_now:.2e}  "
                  f"alpha=[{a[0]:.3f}, {a[1]:.3f}, {a[2]:.3f}, {a[3]:.3f}]")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(
                {
                    "epoch":             epoch,
                    "model_state_dict":  model.state_dict(),
                    "val_loss":          best_val_loss,
                    "train_losses":      train_losses,
                    "val_losses":        val_losses,
                    "alpha_history":     alpha_history,
                    "horizons":          HORIZONS,
                    "max_horizon":       MAX_HORIZON,
                    "latent_dim":        LATENT_DIM,
                    "subject":           subject_id,
                    "fold":              fold_idx + 1,
                },
                ckpt_path,
            )
            patience_counter = 0
        else:
            patience_counter += 1
            if not smoke_test and patience_counter >= patience:
                if verbose:
                    print(f"      Early stopping at epoch {epoch} "
                          f"(no improvement for {patience} epochs).")
                break

    return {
        "subject":       subject_id,
        "fold":          fold_idx + 1,
        "best_val_loss": best_val_loss,
        "train_losses":  train_losses,
        "val_losses":    val_losses,
        "alpha_history": alpha_history,
        "n_train":       len(train_ds),
        "n_val":         len(val_ds),
        "ckpt_path":     ckpt_path,
    }


print("train_one_subject_fold() defined.")

train_one_subject_fold() defined.


---
## 4. Full Experiment: All 16 Subjects × 3 Folds

Runs `train_one_subject_fold` for every subject/fold combination (48 training runs of up
to 100 epochs each, with early stopping). **Intentionally left off**
(`RUN_FULL_TRAINING = False`) until the pipeline has been validated by the smoke test
below — flip the flag to `True` to launch the full protocol. Results accumulate in
`subject_fold_results`, one dict per (subject, fold), and checkpoints land in
`src/models/per_subject_k4/`.

In [6]:
RUN_FULL_TRAINING = False  # set True to launch all 16 subjects x 3 folds (48 runs x up to 100 epochs)

subject_fold_results = []  # one dict per (subject, fold), from train_one_subject_fold()

if RUN_FULL_TRAINING:
    for subject_id in range(1, NUM_SUBJECTS + 1):
        print(f"\nSubject {subject_id}")
        for fold_idx in range(3):
            print(f"  Fold {fold_idx + 1}/3")
            res = train_one_subject_fold(subject_id, fold_idx, epochs=100, verbose=False)
            subject_fold_results.append(res)
            print(f"    -> best val loss = {res['best_val_loss']:.5f}  "
                  f"(saved to {res['ckpt_path']})")
    print("\nPre-training complete for all subjects and folds.")

    all_best = [r["best_val_loss"] for r in subject_fold_results]
    print(f"\nBest val loss across all {len(all_best)} runs: "
          f"{np.mean(all_best):.5f} ± {np.std(all_best):.5f}")
else:
    print("RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run. "
          "See Section 5 for a smoke test of the pipeline instead.")

RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run. See Section 5 for a smoke test of the pipeline instead.


---
## 5. Smoke Test

2-epoch smoke test on **Subject 1, Fold 1** to confirm the per-subject data loading,
fresh-model-per-fold training loop, and checkpointing all work before launching the full
48-run protocol.

In [7]:
print("=" * 70)
print("SMOKE TEST: Subject 1, Fold 1, 2 epochs")
print("=" * 70)

SMOKE_SUBJECT = 1
SMOKE_FOLD    = 0  # Fold 1 (0-based)
SMOKE_EPOCHS  = 2

_train_trials, _val_trials = get_fold_trials(SMOKE_FOLD)
print(f"Train trials ({len(_train_trials)}): {_train_trials}")
print(f"Val trials   ({len(_val_trials)}): {_val_trials}\n")

smoke_result = train_one_subject_fold(
    SMOKE_SUBJECT, SMOKE_FOLD, epochs=SMOKE_EPOCHS, smoke_test=True, verbose=True,
)

print(f"\nTrain pairs: {smoke_result['n_train']:,}   Val pairs: {smoke_result['n_val']:,}")
print(f"Best val loss: {smoke_result['best_val_loss']:.5f}")
print(f"Checkpoint saved to: {smoke_result['ckpt_path']}")
assert os.path.exists(smoke_result["ckpt_path"]), "FAIL: checkpoint was not saved!"
assert smoke_result["best_val_loss"] > 0, "FAIL: val loss should be positive!"
print("\nSMOKE TEST PASSED")

SMOKE TEST: Subject 1, Fold 1, 2 epochs
Train trials (30): [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Val trials   (15): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]



    Epoch   1/2  train=36.18023  val=35.08472  lr=5.27e-04  alpha=[0.143, 0.346, 0.145, 0.366]


    Epoch   2/2  train=30.68937  val=30.82766  lr=0.00e+00  alpha=[0.122, 0.368, 0.122, 0.388]

Train pairs: 902   Val pairs: 561
Best val loss: 30.82766
Checkpoint saved to: /workspaces/AH-PC/src/models/per_subject_k4/ahpc_subject01_fold1_best.pt

SMOKE TEST PASSED


---
## Summary

| Step | Status |
|------|--------|
| Model re-defined: multi-horizon (k=1,2,4,8), Horizon Attention Module + 4 decoder heads | depends on run |
| Per-subject trial-based fold splitting (identical convention to Notebook 04) | depends on run |
| `train_one_subject_fold()`: fresh AHPCModel per (subject, fold) | depends on run |
| Smoke test: Subject 1, Fold 1, 2 epochs | depends on run |
| Full protocol: 48 checkpoints to `src/models/per_subject_k4/` | not launched by default |

**Next step:** with `RUN_FULL_TRAINING = True`, this produces 48 subject-specialised
multi-horizon encoders (`ahpc_subject{sid:02d}_fold{fold}_best.pt` in
`per_subject_k4/`), each pre-trained only on that subject's own two training-session
pairs, routing across k=1,2,4,8 via the Horizon Attention Module. A fine-tuning notebook
analogous to `04d_finetune_per_subject.ipynb` could then load these k=4 per-subject-per-fold
checkpoints (rather than the k=1-only ones in `per_subject/`) before adding the
classification head, to test whether multi-horizon per-subject pre-training improves on
the k=1 per-subject result.